# LC12 — Neural networks, and the discipline of evaluation (self-paced, ~50 min)

Two topics that belong together. First: the multilayer perceptron — the "hello world" of neural networks — applied to the load problem, including the unglamorous detail that decides whether it works at all (feature scaling). Second, and more important for your projects: the **evaluation discipline** that separates results from wishful thinking. Material appears in **Quiz 4**; this is the last taught content before your project period.

**How to use this notebook:** run the cells top to bottom, one at a time. Every code cell has a note above it saying what it is for and what you should see; the numbers quoted are the ones the course series produces. If what you see differs, stop and ask your pod.

## 0. Before you start: opening the notebook on your laptop

Local, not Colab — it reads the course dataset from the repository. From a terminal with your workbook's venv active (the Lab 1 one; `scikit-learn` is in it):

```bash
pip install jupyterlab            # once; the toolbox never imports it, so it is not in requirements.txt
cd <path to your clone of>/course-material/notebooks
jupyter lab                       # opens in your browser; double-click LC12_ann_evaluation.ipynb
```

Start Jupyter *inside* `notebooks/` — the guard cell checks that. Shift+Enter runs a cell; the kernel offered, "Python 3 (ipykernel)", is your venv's Python when Jupyter was started from the activated venv. The whole notebook runs in about a minute.

### 0.1 Setup

`%pip install` installs into the notebook's own Python; in the course venv everything is already there at its pinned version. **What you see:** one "Note: you may need to restart the kernel…" line, or nothing.

In [1]:
# Install exactly what this notebook uses.
%pip install scikit-learn pandas numpy pyarrow matplotlib --quiet


Note: you may need to restart the kernel to use updated packages.


The guard: `assert` stops here with a clear message if `../data` is missing. **What you see:** nothing — good.

In [2]:
from pathlib import Path
# Guard: this notebook expects to run from the notebooks/ folder of a clone of
# the course repository — the datasets live one level up in ../data/.
# Failing here, early and clearly, beats a confusing FileNotFoundError later.
assert Path("../data").exists(), (
    "Course data folder not found. Clone KTH-EG2140/course-material and open "
    "this notebook from its notebooks/ folder.")

### 0.2 The table, cut in three

LC10's feature table, with one change: the split has three parts instead of two. Training stops at the end of August, September is the *validation* set, October onward the *test* set — section 2 says what each is for. **What you see:** `train 5664 | validation 720 | test 2208 — three sets, three jobs`.

In [3]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
df = pd.read_parquet("../data/svedala-year/svedala_hourly.parquet")
y = df["ZON_MITT"].interpolate(limit=3)
X = pd.DataFrame(index=df.index)                 # the LC10 feature table again
X["lag24"], X["lag168"] = y.shift(24), y.shift(168)
X["temp"] = df["temp_mid"].interpolate(limit=3)
X["hour_sin"] = np.sin(2*np.pi*df.index.hour/24); X["hour_cos"] = np.cos(2*np.pi*df.index.hour/24)
X["dow"] = df.index.dayofweek; X["workday"] = (X["dow"] < 5).astype(int)
data = X.join(y.rename("target")).dropna()
# THREE sets this time: train up to August, validation = September, test = October on
train = data.loc[:"2025-08-31"]; val = data.loc["2025-09-01":"2025-09-30"]; test = data.loc["2025-10-01":]
Xtr, ytr = train.drop(columns="target"), train["target"]
Xva, yva = val.drop(columns="target"), val["target"]
Xte, yte = test.drop(columns="target"), test["target"]
print(f"train {len(Xtr)} | validation {len(Xva)} | test {len(Xte)} — three sets, three jobs")

train 5664 | validation 720 | test 2208 — three sets, three jobs


## 1. The MLP — and where the scaler really earns its keep

A neural network is layers of weighted sums squeezed through nonlinearities, trained by gradient descent. Textbooks say "always scale your features" — and it costs nothing, so we do. How much it matters depends on data and configuration: anywhere from barely at all to a total collapse of training. On this data, with these settings, it is worth about 5 % of MAE — modest — and you never know in advance which regime you are in, which is exactly why the habit is unconditional.

The deeper reason the `StandardScaler` sits inside a **pipeline** is not speed — it is *honesty*: the scaler is fitted on training data only and travels with the model, so the test set's statistics can never leak into preprocessing.

### 1.1 Two identical networks, one scaled

`MLPRegressor(hidden_layer_sizes=(32,16))` is a network with two hidden layers of 32 and 16 units; `max_iter=300` caps the training passes; `random_state=0` pins the random starting weights so your numbers match these. The second model is a **pipeline**: `make_pipeline(StandardScaler(), MLPRegressor(...))` chains two steps, so `.fit` first learns each feature's mean and spread on the training rows and rescales them to zero mean and unit spread, then trains the network on the rescaled table; `.predict` applies the same rescaling before asking the network. Both are scored on the *validation* set, not the test set. The two fits take about ten seconds together.

**What you see:**

```
MLP unscaled  val MAE:   107.9 MW
MLP + scaler  val MAE:   102.5 MW
```

A 5 MW difference — real, not dramatic. The features here already sit within a factor of ten of each other (loads in the thousands, temperatures in the tens), which is why the unscaled network survives; feed it a feature in the millions next to one in units and it does not.

In [4]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
raw = MLPRegressor(hidden_layer_sizes=(32,16), max_iter=300, random_state=0).fit(Xtr, ytr)
scaled = make_pipeline(StandardScaler(),
         MLPRegressor(hidden_layer_sizes=(32,16), max_iter=300, random_state=0)).fit(Xtr, ytr)
mae_raw = float(np.abs(raw.predict(Xva)-yva).mean())
mae_scaled = float(np.abs(scaled.predict(Xva)-yva).mean())
print(f"MLP unscaled  val MAE: {mae_raw:7.1f} MW")
print(f"MLP + scaler  val MAE: {mae_scaled:7.1f} MW")
print("-> scaling pays a little here, and would pay a lot with wilder feature ranges;\n"
      "   the pipeline additionally guarantees that no test-set statistics leak into the preprocessing")

MLP unscaled  val MAE:   107.9 MW
MLP + scaler  val MAE:   102.5 MW
-> scaling pays a little here, and would pay a lot with wilder feature ranges;
   the pipeline additionally guarantees that no test-set statistics leak into the preprocessing


Same architecture, a different outcome — preprocessing IS part of the model, and its cost is one line. (This is also why the pipeline object exists: the scaler is fitted on training data only and travels with the model, so the test set never leaks into the preprocessing.)

## 2. Train / validation / test — three sets, three jobs

- **Train**: fit parameters.
- **Validation**: choose between models and settings. You may look as often as you like — and every look costs a little honesty, which is why the third set exists.
- **Test**: touched ONCE, at the end, to report the number. A test set consulted during development is just a second validation set wearing a costume.

Model selection happens on validation:

### 2.1 Select on validation, then touch the test set once

Three candidates: the scaled MLP from above, a wider one (64 and 32 units — a second fit of about ten seconds), and LC10's gradient boosting. `{k: ... for k, m in candidates.items()}` is a dictionary comprehension scoring each on *validation*. `min(val_mae, key=val_mae.get)` returns the name with the smallest MAE — the winner is chosen without the test set ever being consulted.

**What you see:**

```
gbdt            84.6
MLP (64,32)     98.8
MLP (32,16)    102.5

selected on validation: gbdt — ONLY this one may now see the test set
```

Gradient boosting wins September by a wide margin. The MLPs are not bad models; on 5 664 training rows the tree ensemble simply has the easier job.

In [5]:
from sklearn.ensemble import HistGradientBoostingRegressor
# Three candidates fitted on TRAIN only; the dict maps display name -> model.
# Selection among them happens on VALIDATION — the test set stays untouched.
candidates = {
    "MLP (32,16)": scaled,
    "MLP (64,32)": make_pipeline(StandardScaler(),
        MLPRegressor(hidden_layer_sizes=(64,32), max_iter=300, random_state=0)).fit(Xtr, ytr),
    "gbdt": HistGradientBoostingRegressor(random_state=0).fit(Xtr, ytr),
}
val_mae = {k: float(np.abs(m.predict(Xva)-yva).mean()) for k, m in candidates.items()}
print(pd.Series(val_mae).round(1).sort_values().to_string())
winner = min(val_mae, key=val_mae.get)
print(f"\nselected on validation: {winner} — ONLY this one may now see the test set")

gbdt            84.6
MLP (64,32)     98.8
MLP (32,16)    102.5

selected on validation: gbdt — ONLY this one may now see the test set


The one permitted look at the test set: the selected model, scored on October, next to persistence. **What you see:** `reported test MAE (gbdt): 132.2 MW   [persistence: 178.5 MW]`. Note the test MAE (132) is much larger than the validation MAE (85): September was an easy month, October is not — a reminder that the validation number is for *choosing*, and only the test number is for *reporting*.

In [6]:
final = float(np.abs(candidates[winner].predict(Xte)-yte).mean())
persist = float(np.abs(Xte["lag24"]-yte).mean())
print(f"reported test MAE ({winner}): {final:.1f} MW   [persistence: {persist:.1f} MW]")

reported test MAE (gbdt): 132.2 MW   [persistence: 178.5 MW]


## 3. The leakage gallery — how good numbers lie

The most common ways an evaluation flatters itself, all seen in real student and industry work:

1. **Random split of a time series** — tomorrow's neighbours land in the training set; the model "predicts" what it has effectively seen.
2. **Scaling fitted on all data** — the test set's statistics leak into preprocessing (the pipeline above prevents exactly this).
3. **Feature built with future information** — a rolling mean centred on t uses t+1; a daily mean feature computed over the whole day "predicts" 03:00 using 23:00.
4. **Test-set shopping** — trying weeks until the number looks good (the Lab 7 rule returns).
5. **Target leakage** — a feature that is the answer in disguise (total_mw as a feature for ZON_MITT).

Every one of these produces a *better-looking* number. That is precisely why they must be hunted deliberately — nothing in the code smells wrong.

## 4. Choosing the metric is choosing what matters

MAE treats all MW equally; RMSE punishes large misses harder (peak errors matter more to an operator); MAPE breaks near zero and flatters high-load hours; pinball scores quantiles. Your project must *choose and justify* — "we used MAE" is a decision, and it belongs in the decision log.

## Self-check

Three promises: scaling did not hurt the MLP, neither MLP variant collapsed (their MAEs are within a factor of three — a guard against a training run that diverged), and the selected model beats persistence on the untouched test set. **What you see:** `ALL OK — models done, discipline installed. …`

In [7]:
assert mae_scaled <= mae_raw, "scaling should not make the MLP worse"
assert max(mae_raw, mae_scaled) < 3 * min(mae_raw, mae_scaled), "one MLP variant collapsed unexpectedly"
assert final < persist, "the selected model must beat persistence on the untouched test set"
print("ALL OK — models done, discipline installed. Module 4 hands you an AI assistant; "
      "everything in this notebook is what keeps you the engineer in the room.")

ALL OK — models done, discipline installed. Module 4 hands you an AI assistant; everything in this notebook is what keeps you the engineer in the room.


## 5. What you take to Labs 8 and 9

Lab 9's challenger — the MLP if that is your pair's claim — goes into Lab 7's harness inside a pipeline exactly like the one above, so the scaler is refitted on each day's training window and never sees the day being forecast. Lab 8 splits its labelled hours temporally, chooses its threshold on validation and reports on test. And the gallery of section 3 is the checklist you run on your own results before you believe them.